# NutriEpiDB: A Database For Dietary Compounds With Epigenetic Targets Exploration with `mlcroissant`

This notebook provides a step-by-step guide to loading, exploring, and analyzing the NutriEpiDB dataset using the [`mlcroissant`](https://mlcroissant.org) library.

### Dataset Source
The dataset is described by a Croissant schema and can be accessed at the following URL:

https://sen.science/doi/10.71728/senscience.sx3s-9110/fair2.json


In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. This will extract structured information about dietary compounds, food sources, epigenetic targets, and associated metadata.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.sx3s-9110/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (as an object, not a dict)
metadata = dataset.metadata
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Keywords: {metadata.keywords}")


## 2. Data Overview

Review available record sets and their fields. We will reference all entities by their `@id` identifiers to ensure consistent access and manipulation, as recommended for Croissant datasets. This step helps us discover the main data tables and their schema.


In [ ]:
# List all record sets by @id
record_sets = dataset.metadata.recordSet if hasattr(dataset.metadata, 'recordSet') else []
if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    for record_set in record_sets:
        print(f"Record Set @id: {record_set['@id']}")
        if 'field' in record_set:
            fields = record_set['field']
            field_ids = [f["@id"] if isinstance(f, dict) and '@id' in f else f for f in fields]
            print(f"  Fields @id: {field_ids}")
        else:
            print("  No fields defined in this record set.")

## 3. Data Extraction

Load data from the main record sets into pandas DataFrames for analysis. All entities are referenced by their `@id` as per the schema. We'll extract a preview and column names for review.


In [ ]:
# Extract data from each record set using their @id
dataframes = {}
# Prepare a list of recordSet @id values
record_set_ids = []
if hasattr(dataset.metadata, 'recordSet'):
    for record_set in dataset.metadata.recordSet:
        record_set_ids.append(record_set['@id'])

for rset_id in record_set_ids:
    # Use the mlcroissant API to load records by record_set @id
    records = list(dataset.records(record_set=rset_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rset_id] = df

# Show available DataFrames and their column names
for rset_id in dataframes:
    print(f"Columns in record set {rset_id}:", dataframes[rset_id].columns.tolist())
    display(dataframes[rset_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply some basic processing steps: filtering, normalizing, and grouping. We'll reference numeric and grouping fields by their respective `@id`s. Let's choose a relevant record set (for illustration, pick the first available record set) and analyze a numeric field (such as binding affinity measures).


In [ ]:
# Select record set and field @ids for analysis
# We'll look for columns named similar to 'binding_affinity' or 'affinity' as described in the schema.
record_set_id = list(dataframes.keys())[0] if dataframes else None
if record_set_id:
    df = dataframes[record_set_id]
    numeric_field = None
    group_field = None
    # Try to find likely fields
    for col in df.columns:
        if 'affinity' in col.lower():
            numeric_field = col
        if 'compound' in col.lower():
            group_field = col

    if numeric_field:
        # Filter records with affinity above a sample threshold (e.g., 10)
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by compound type if available
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric affinity field found. Columns:", df.columns.tolist())
else:
    print("No dataframes available for EDA.")

## 5. Visualization

Visualize the distribution of binding affinity or other relevant numeric field. We'll use matplotlib for plotting. All references are via their `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field} in record set {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    # If grouped_df was defined
    if 'grouped_df' in locals():
        plt.figure(figsize=(8,5))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df.head(20))
        plt.title(f"Mean {numeric_field} by {group_field} (Top 20)")
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("Unable to plot: numeric field or dataframe not available.")

## 6. Conclusion

- Using `mlcroissant`, we've loaded and explored the NutriEpiDB dataset, referencing all elements by their `@id` as recommended for FAIR data.
- We reviewed the structure, loaded record sets, previewed fields, and performed basic filtering, normalization, grouping, and visualization.
- The dataset provides structured insights into dietary compounds, food sources, epigenetic targets, and their interactions, supporting nutrigenomics and biomarker discovery research.
- For detailed applications, consult the NutriEpiDB documentation or access additional fields and relationships via their `@id`s.
